# 01 — Data Preparation

Cleans the raw Spotify dataset and extracts playlist plays into tabular form. Downstream feature selection (02) and feature engineering (03) consume these two files.

**Inputs** (`data/`):
- `data/Spotify_dataset_gigasheet.csv` — raw Spotify audio-feature catalogue
- `data/playlist.json` — raw Million Playlist Dataset slice (1,000 playlists)

**Outputs** (`outputs/`):
- `outputs/spotify_cleaned.csv` — cleaned catalogue (19,675 songs × 25 cols)
- `outputs/USETHIS_output_filtered.csv` — playlist plays whose tracks appear in the catalogue (15,011 rows)

# 1. Data Cleaning


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df = pd.read_csv("./data/Spotify_dataset_gigasheet.csv")
df

In [ ]:
print((df['Title'] == 'None').sum())
df.isna().sum()

In [ ]:
df['Album_type'].value_counts()

In [ ]:
#remove compilations
df = df[(df['Album_type'] != "compilation")].copy()

In [ ]:
df['Duration_min'].sort_values()

In [ ]:
# remove tracks longer than 10 mins
df = df[(df['Duration_min'] < 10)].copy()

In [ ]:
df = df.sort_values('Stream', ascending=False)
df.duplicated(subset=['Artist', 'Track']).value_counts()

In [ ]:
# remove duplicate artist/track combos
df = df.drop_duplicates(subset=['Artist', 'Track'], keep='first')

In [ ]:
numerical_cols = df.select_dtypes(include='number').columns
zero_counts = (df[numerical_cols] == 0).sum()
print(zero_counts)

In [ ]:
#remove rows with zero tempo and duration
df = df[(df['Tempo'] != 0) & (df['Duration_min'] != 0)].copy()

# set flag for zero views and streams
df['zero_engagement_flag'] = (df['Views'] == 0) | (df['Stream'] == 0)
df

In [ ]:
df.to_csv('outputs/spotify_cleaned.csv', index=False)

# 2. Playlist Extraction

Ports `archive/Playlist Cleaning.ipynb`:

1. **Unique-track whitelist** from the cleaned catalogue (archive cell 0 — originally read `cleandata.csv` and exported `artist_track.csv`; we build it from `spotify_cleaned.csv` produced in §1).
2. **Flatten `playlist.json`** into `(pid, artist_name, track_name)` rows (archive cell 1).
3. **Filter** the flattened extract against the whitelist (archive cell 2 — originally filtered against a separate `unique_tracks.csv`).

The filter here is redundant with the inner-join in §4 (both drop plays whose tracks are not in the catalogue), but it is kept for provenance and produces the same final `taste_profiles.csv` / `user_liked_songs.csv`.

In [ ]:
import json
import csv
import pandas as pd

# Step 1 (archive cell 0): build unique (artist, track) whitelist from the cleaned catalogue
catalogue = pd.read_csv('outputs/spotify_cleaned.csv', usecols=['Artist', 'Track'])
unique_tracks = set(zip(catalogue['Artist'].astype(str).str.strip(),
                        catalogue['Track'].astype(str).str.strip()))
print(f'Unique-track whitelist: {len(unique_tracks)} (artist, track) pairs')

# Step 2 (archive cell 1): flatten playlist.json -> (pid, artist_name, track_name)
with open('data/playlist.json') as f:
    data = json.load(f)

raw_rows = [
    {'pid': p['pid'], 'artist_name': t['artist_name'], 'track_name': t['track_name']}
    for p in data['playlists'] for t in p['tracks']
]
playlists_raw = pd.DataFrame(raw_rows)
print(f'Extracted {len(playlists_raw)} track-plays from {playlists_raw["pid"].nunique()} playlists '
      f'(avg {playlists_raw.groupby("pid").size().mean():.2f} songs per playlist)')

# Step 3 (archive cell 2): keep only plays whose (artist, track) is in the whitelist
mask = [(a.strip(), t.strip()) in unique_tracks
        for a, t in zip(playlists_raw['artist_name'], playlists_raw['track_name'])]
playlists_filtered = playlists_raw[mask].reset_index(drop=True)
removed = len(playlists_raw) - len(playlists_filtered)
print(f'Kept {len(playlists_filtered)} rows, removed {removed} '
      f'({removed/len(playlists_raw)*100:.1f}%) not in catalogue')

playlists_filtered.to_csv('outputs/USETHIS_output_filtered.csv', index=False)
print('Saved outputs/USETHIS_output_filtered.csv')